In [1]:
%pip install mlflow -qq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 787.0/787.0 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 60.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
!uv pip install -q gdown
!gdown --id 1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS -O dataset.zip
!unzip -q dataset.zip

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS
From (redirected): https://drive.google.com/uc?id=1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS&confirm=t&uuid=fecdb6d2-133a-4c2e-b33b-a113e634c7be
To: /kaggle/working/dataset.zip
100%|████████████████████████████████████████| 356M/356M [00:06<00:00, 53.6MB/s]


In [3]:
import sys

sys.path.append('/kaggle/input/datasets/maksimbessolitsyn/')

In [4]:
%pip install mlflow -qq

Note: you may need to restart the kernel to use updated packages.


In [5]:
import logging
import warnings
import os

warnings.filterwarnings("ignore", category=UserWarning, module=r"torch(\.|$)")
warnings.filterwarnings("ignore", category=FutureWarning, module=r"torch(\.|$)")
logging.getLogger("torch").setLevel(logging.ERROR)
logging.getLogger("torch._dynamo").setLevel(logging.ERROR)




In [6]:
LOG_DIR = "./mlruns"


In [7]:
DATA_DIR = "."
PATH_INTERACTIONS = os.path.join(DATA_DIR, "interactions.parquet")
PATH_EMBEDDINGS = os.path.join(DATA_DIR, "embeddings.parquet")
PATH_ARTISTS = os.path.join(DATA_DIR, "artists.parquet")
SEED = 42

In [8]:
from sasrec import run_ddp_training, ExperimentConfig

In [9]:
fixed_experiment_parameters = ExperimentConfig(
    graph=ExperimentConfig.GraphConfig(
        n_layers=4,
        d_model=256,
        n_heads=4,
        dropout=0.0,
        log_q_correction=1.0,
        is_cosine_similarity=True,
    ),
    data=ExperimentConfig.DataConfig(
        vocab_size=157_162,
        max_seq_len=100,
        bos=0,
        path_interactions=PATH_INTERACTIONS,
        path_embeddings=PATH_EMBEDDINGS,
        path_artists=PATH_ARTISTS,
        core_min_interaction_per_user=5,
        test_interval_seconds=7 * 24 * 60 * 60,
        max_train_events_per_user=100,
    ),
    tau=None,
    training_dataset=None,
    test_dataset=ExperimentConfig.TestDatasetConfig(
        batch_size=32,
        device="cuda",
    ),
    optimizer=None,
    scheduler=ExperimentConfig.SchedulerConfig(
        class_name=None,
        json_args={},
    ),
    training=ExperimentConfig.TrainingConfig(
        num_epochs=15,
        grad_clip=1.0,
        eval_every=1,
        logging=True,
        log_dir=LOG_DIR,
        seed=SEED,
    ),
    evaluator=ExperimentConfig.EvaluatorConfig(
        topk=100,
    ),
)

In [10]:
from dataclasses import replace
import torch

tau = ExperimentConfig.TauConfig(
    class_name="ConstantTau",
    json_args={
        "initial_tau": None,
        "tau_min": None,
        "tau_max": None,
        "num_epochs": None,
        "num_tokens_per_epoch": None,
    },
)

training_dataset = ExperimentConfig.TrainingDatasetConfig(
    batch_size=128,
    device="cuda",
    chunk_rows=64000,
    shuffle=True,
    seed=42,
    pin_memory=True,
    uniform_negative_items=None,
    in_batch_negative_items=None,
)

optimizer = ExperimentConfig.OptimizerConfig(
    class_name="AdamW",
     json_args={
        "lr": 3e-3,
        "weight_decay": 1e-5,
    },
)

for initial_tau in [0.04, 0.045, 0.05]:
    print(f"Running experiment with {initial_tau=}...")

    tau.json_args["initial_tau"] = initial_tau

    for uniform, unigram in [(18_000, 12_000), (22_000, 8_000), (26_000, 4_000)]:
        run_ddp_training(
            replace(
                fixed_experiment_parameters, 
                tau=tau,
                training_dataset=replace(
                    training_dataset,
                    uniform_negative_items=uniform, 
                    in_batch_negative_items=unigram,
                ),
                optimizer=optimizer
            ),
            world_size=torch.cuda.device_count()
        )


Running experiment with initial_tau=0.04...


Epochs: 100%|██████████| 15/15 [29:51<00:00, 119.46s/it, train_loss=8.0613]


--------------------------------
Experiment name: Constant[value=0.04]
hitrate: 0.3556
recall: 0.1210
ndcg: 0.0486
coverage: 0.4123
--------------------------------


Epochs: 100%|██████████| 15/15 [30:03<00:00, 120.25s/it, train_loss=7.5495]


--------------------------------
Experiment name: Constant[value=0.04]
hitrate: 0.3606
recall: 0.1232
ndcg: 0.0491
coverage: 0.4000
--------------------------------


...

--------------------------------
Experiment name: Constant[value=0.05]
hitrate: 0.3588
recall: 0.1232
ndcg: 0.0499
coverage: 0.3335
--------------------------------


Epochs: 100%|██████████| 15/15 [30:00<00:00, 120.03s/it, train_loss=7.8758]


--------------------------------
Experiment name: Constant[value=0.05]
hitrate: 0.3603
recall: 0.1257
ndcg: 0.0514
coverage: 0.3541
--------------------------------


Epochs: 100%|██████████| 15/15 [30:05<00:00, 120.36s/it, train_loss=7.5372]


--------------------------------
Experiment name: Constant[value=0.05]
hitrate: 0.3633
recall: 0.1253
ndcg: 0.0504
coverage: 0.4136
--------------------------------
